# Imports and Graphical Setups

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.feature_selection import f_classif
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
import math


from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
plt.rcParams['lines.linewidth'] = 3
plt.rcParams['figure.figsize'] = [8, 5]
plt.rcParams['font.size'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 20
# plt.rcParams.keys()
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut, StratifiedKFold,KFold
from statistics import mean

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Data Analysis

## 1.**Descriptive Statistics**

In [ ]:
df = pd.read_csv('../datasets/group_14.csv')
df.describe()

In [ ]:
df.select_dtypes(include="object").head(10)

It is identified that 'focus_factor' is considered an object (string) rather than a float.
This transformation converts it into a proper numeric type.
The substitution of ',' with '.' is done to standardize decimal representation
and make numeric conversion possible.


In [ ]:
# Convert 'focus_factor' values to string to ensure consistency
df["focus_factor"] = (
    df["focus_factor"]
    .astype(str)                    # Ensure all values are strings
    .str.replace(",", ".", regex=False)  # Replace commas with dots for decimal conversion
)

# Convert the cleaned strings to numeric values (floats)
# Invalid parsing will be set to NaN (errors='coerce')
df["focus_factor"] = pd.to_numeric(df["focus_factor"], errors="coerce").astype(float)

Check for Missing Values and Unique Values
This shows that there is no missing values found in the dataset

In [ ]:
data_info = pd.DataFrame({
    'Data Type': df.dtypes,
    'Missing Values': df.isnull().sum(),
    'Unique Values': df.nunique()
})
data_info

Check Duplicates

In [ ]:
duplicated = df.duplicated()
print(df[duplicated].shape)

## 2.**Univariate analysis**


In [ ]:
sns.set_theme(style='whitegrid')
num_cols = df.select_dtypes(include='number').columns
n = len(num_cols)

df_melt = df.melt(value_vars=num_cols)
g = sns.FacetGrid(df_melt, col="variable", col_wrap=5, sharex=False, sharey=False, height=3)
g.map(sns.histplot, "value", kde=True, bins=20)
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
fig.suptitle('Distribution of Categorical Variables', fontsize=16)

sns.countplot(x='target_class', data=df, ax=ax)
ax.set_title('target_class')

plt.tight_layout()
plt.show()

## 3.**Bivariate Analysis**


In [ ]:
num_cols = df.select_dtypes(include='number').columns.drop('target_regression')

# Reshape the dataframe for FacetGrid plotting
df_melt = df.melt(id_vars='target_regression', value_vars=num_cols,
                  var_name='Feature', value_name='Value')

# Create scatterplots of each feature vs target
g = sns.FacetGrid(df_melt, col="Feature", col_wrap=4, sharex=False, sharey=False, height=3)
g.map_dataframe(sns.scatterplot, x="Value", y="target_regression", alpha=0.6)
g.set_titles(col_template="{col_name}")
g.set_axis_labels("Feature Value", "Target (Regression)")
plt.tight_layout()
plt.show()

In [ ]:
# All feature columns (exclude target_class)
feature_cols = [col for col in df.columns if col != 'target_class']

n_features = len(feature_cols)
n_cols = 4                     # 4 plots per row
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4 * n_rows))
axes = axes.flatten()

fig.suptitle('Box Plots of Features vs Target Class', fontsize=16, y=1.02)

for ax, feature in zip(axes, feature_cols):
    sns.boxplot(x='target_class', y=feature, data=df, ax=ax)
    ax.set_title(f'{feature} vs target_class')
    ax.set_xlabel('target_class')
    ax.set_ylabel(feature)

# Hide any remaining unused subplots
for i in range(len(feature_cols), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
df_encoded = pd.get_dummies(data=df, columns=["target_class"], drop_first=True)

corr_matrix = df_encoded.corr().abs()

# Set up the figure — make it large enough for 40+ features
plt.figure(figsize=(20, 16))

# Draw the heatmap
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",       # visually clear color map
    annot=False,           # turn off annotations for readability
    linewidths=0.5,        # thin lines between cells
    cbar_kws={"shrink": 0.7},  # make colorbar smaller
    square=True
)

# Improve layout and label rotation for readability
plt.title("Correlation Matrix (Absolute Values)", fontsize=16, pad=20)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:

# Unstack and filter upper triangle only
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# List pairs above a threshold
threshold = 0.8
high_corr_pairs = (
    upper_triangle.stack()
    .reset_index()
    .rename(columns={'level_0': 'Feature 1', 'level_1': 'Feature 2', 0: 'Correlation'})
    .query(f'Correlation > {threshold}')
    .sort_values(by='Correlation', ascending=False)
)

print("Highly correlated feature pairs (|r| > 0.8):")
print(high_corr_pairs)

Some features in the dataset are highly correlated (|r| > 0.8), meaning they carry almost identical information. Including multiple highly correlated features in a regression model can cause multicollinearity, making coefficients unstable and reducing interpretability.

To address this, we retain only one representative from each highly correlated group — typically the most interpretable or relevant feature — and exclude the redundant ones.

**Excluded features (highly correlated/redundant):**
`signal_power, temp_zscore, duration_log_z, energy_rank_pct, tempo_vs_genre, intensity_level, happy_dance`

In [ ]:
# Compute correlation of all columns with target
target_corr = df_encoded.corr()[['target_regression']].sort_values(by='target_regression', ascending=False)

# Plot as a heatmap
plt.figure(figsize=(6, len(target_corr) * 0.5))  # adjust height for many features
sns.heatmap(target_corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation of Features with target_regression")
plt.show()

The features "is_dance_hit" and "echo_constant" show no measurable correlation (NaN or near zero)
 with the target variable "target_regression".

 This suggests they provide little to no linear predictive value for the regression model.

# Regression

## Simple Regression

In [ ]:
features = [col for col in df_encoded.columns if col != "target_regression"]
resultados = []

y = df_encoded["target_regression"]

for feature in features:
    x = df_encoded[[feature]]

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    slr = LinearRegression().fit(x_train, y_train)
    y_pred = slr.predict(x_test)

    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    resultados.append({"feature": feature, "R2": r2, "MSE": mse, "MAE": mae})


df_resultados = pd.DataFrame(resultados).sort_values(by="R2", ascending=False)

print(df_resultados)

In [ ]:

features = [col for col in df_encoded.columns if col != "target_regression"]
resultados = []

y = df_encoded["target_regression"]

for feature in features:
    x = df_encoded[[feature]]

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    slr = LinearRegression().fit(x_train, y_train)
    y_pred = slr.predict(x_test)

    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    slr_error = y_test - y_pred
    df_encoded['slr_error'] = slr_error

    resultados.append({"feature": feature, "R2": r2, "MSE": mse, "MAE": mae})

    # # Gráfico de regressão
    # plt.figure()
    # plt.scatter(x_test, y_test, color="blue", label="Real")
    # plt.plot(x_test, y_pred, color="red", label="Regressão Linear")
    # plt.title(f"Regressão Linear - {feature}")
    # plt.xlabel(feature)
    # plt.ylabel("target_regression")
    # plt.legend()
    # plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Gráfico de dispersão
    axes[0].plot(x, y, 'bo', label='Valores Reais')
    axes[0].plot(x_test, y_pred, 'go', label='Valores Preditos')
    axes[0].set_title("Dispersão: Reais vs. Preditos")
    axes[0].set_xlabel(feature)
    axes[0].set_ylabel("target_regression")
    axes[0].legend()

    # Gráfico de distribuição (usando kdeplot)
    sns.kdeplot(y, color="b", label="Valores Reais", ax=axes[1])
    sns.kdeplot(y_pred, color="g", label="Valores Preditos", ax=axes[1])
    axes[1].set_title("Distribuição: Reais vs. Preditos")
    axes[1].legend()

    # Gráfico de erro de predição
    sns.scatterplot(x=y.index, y='slr_error', data=df_encoded, color="r", ax=axes[2])
    axes[2].set_title("Erro de Predição")
    axes[2].set_ylabel("Erro de Predição")

    fig.tight_layout()
    plt.show()

df_resultados = pd.DataFrame(resultados).sort_values(by="R2", ascending=False)
print(df_resultados)

O melhor feature, segundo o R², é **artists_avg_popularity**, pois possui o maior **valor de R² (0.629417)**. Isso indica que ele explica melhor a variabilidade do alvo na regressão, comparado aos outros. Além de ele <u>também apresentar os menores valores de MSE e MAE</u>, reforçando sua qualidade como preditor.

## Multiple Regression

In [ ]:
# Encode categorical variable
df_encoded = pd.get_dummies(data=df, columns=["target_class"], drop_first=True)

y = df_encoded['target_regression']
X = df_encoded.drop(columns=["target_regression"])

Demonstrate Distribution and Residuals graph


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
lr = LinearRegression()
lr_model = lr.fit(X_train, y_train)

predictions = lr_model.predict(X_test)
train_predictions = lr_model.predict(X_train)

# Create DataFrame for plotting
df_plot = pd.DataFrame({
    'pr_result': predictions,
    'pr_error': y_test - predictions
})

# Create figure with 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- (1) Distribution plot: Actual vs Predicted ---
sns.kdeplot(y_test, color="g", label="Actual Values", ax=axes[0])
sns.kdeplot(df_plot['pr_result'], color="r", label="Predicted Values", ax=axes[0])
axes[0].set_title("Distribution: Actual vs Predicted Values", fontsize=12)
axes[0].legend()
axes[0].set_xlabel("Target Value")
axes[0].set_ylabel("Density")

# --- (2) Prediction error (residuals) scatter plot ---
sns.scatterplot(x=df_plot.index, y='pr_error', data=df_plot, color="r", ax=axes[1])
axes[1].set_title("Prediction Error (Residuals)", fontsize=12)
axes[1].set_ylabel("Prediction Error")
axes[1].set_xlabel("Sample Index")
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)  # reference line at 0
axes[1].grid(True, linestyle=':', linewidth=0.7)

plt.tight_layout()
plt.show()

print("MAE (Train):", mean_absolute_error(y_train, train_predictions))
print("MAE (Test):", mean_absolute_error(y_test, predictions))
print("R² (Train):", r2_score(y_train, train_predictions))
print("R² (Test):", r2_score(y_test, predictions))


In [ ]:
# --- Set Seaborn theme ---
sns.set_context("talk")  # larger fonts for readability

# --- Use TEST DATA (correct for evaluation plots) ---
X_plot = X_test.copy()
y_plot = y_test.copy()
predictions_plot = predictions

# Select only numerical features
num_features = X_plot.select_dtypes(include='number').columns.tolist()

# Create subplots grid
n = len(num_features)
cols = 4
rows = (n + 2) // cols + ((n + 2) % cols > 0)  # +2 for dist & residual plots

fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 4))
axes = axes.flatten()

# --- Scatter plots: actual vs predicted per feature ---
for i, feature in enumerate(num_features):
    sns.scatterplot(
        x=X_plot[feature], y=y_plot,
        color='royalblue', label='Actual', s=60, alpha=0.6, ax=axes[i]
    )
    sns.scatterplot(
        x=X_plot[feature], y=predictions_plot,
        color='limegreen', label='Predicted', s=60, alpha=0.6, ax=axes[i]
    )
    axes[i].set_title(f"Actual vs Predicted: {feature}", fontsize=14, weight='bold')
    axes[i].set_xlabel(feature, fontsize=12)
    axes[i].set_ylabel("Target", fontsize=12)
    axes[i].legend()
    axes[i].grid(True, linestyle='--', alpha=0.5)


# Remove any unused axes
for j in range(n , len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Analysis

The list features_r2_scores ranks predictors by their individual explanatory power (R²) in simple linear regression with the target variable. The list features_to_exclude contains predictors identified through global analysis as having little to no correlation with the target variable and high correlation with other predictors.

In [ ]:

#r2_scores sorted
features_r2_scores = ['artists_avg_popularity','album_freq','target_class_class_65','purity_score','signal_strength','signal_power','energy_rank_pct','loudness_yeo','intensity_level','duration_4','duration_log','duration_log_z','popularity_level','target_class_class_73','loud_energy_ratio','mood_pca','loudness_level','duration_2','mode_indicator','loudness_intensity','artist_song_count','explicit','temp_zscore','activity_rate','happy_dance','positivity_index', 'tempo_class','duration_3','duration_1','acoustic_valence_mood_cluster','duration_5','time_signature_class_boolean','key_mode','time_signature','acoustics_instrumental','timbre_index','resonance_factor','movement_index','mood_cluster','ambient_level','key_sin','key_cos','verbal_density','tempo_vs_genre','focus_factor','distorted_movement','is_instrumental','is_dance_hit','echo_constant']

#Possible features to Exclude
features_to_exclude= [
    "signal_power","temp_zscore","duration_log_z","energy_rank_pct","tempo_vs_genre","intensity_level","happy_dance""is_dance_hit""echo_constant"
]


In [ ]:
# Encode categorical variable
df_encoded = pd.get_dummies(data=df, columns=["target_class"], drop_first=True)

# Define target and features
y = df_encoded['target_regression']
X = df_encoded.drop(columns=["target_regression"])

seed = 42

X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
)

We performed a **sequential forward feature selection** to iteratively add features that improve the model's performance.

- **Starting point:** The top 2 features based on individual R² scores, ensuring a reasonable baseline.
- **Improvement threshold (0.001):** Chosen to allow small but meaningful gains; a higher threshold (e.g., 0.01) would have kept only the initial two features.
- **Decrease threshold (0.01):** Stops the process if adding a feature significantly reduces train R², preventing deterioration.
- **Approach:** At each step, we temporarily add a feature, evaluate the train R², and decide to **keep**, **ignore**, or **stop** based on the thresholds.

In [ ]:
features_to_verify = [f for f in features_r2_scores if f not in features_to_exclude]

# Start with top 2 features
selected_cols = features_to_verify[:2]
X_current = X_train[selected_cols].copy()

lr = LinearRegression()
lr.fit(X_current, y_train)
y_pred_train = lr.predict(X_current)
prev_r2 = r2_score(y_train, y_pred_train)

print(f"Starting with {selected_cols}, initial train R²: {prev_r2:.5f}\n")

# Thresholds
improvement_threshold = 0.001
decrease_threshold = 0.01

# List to store results of each iteration
log_records = []

# Forward selection loop (sequential, not full forward)
for col in features_to_verify[2:]:
    if col not in X_train.columns:
        print(f"Skipped '{col}' (not found in data)")
        continue

    X_temp_train = pd.concat([X_current, X_train[[col]]], axis=1)
    lr.fit(X_temp_train, y_train)
    y_pred_train = lr.predict(X_temp_train)
    new_r2 = r2_score(y_train, y_pred_train)

    delta = new_r2 - prev_r2

    # Log each attempt
    log_records.append({
        "Feature": col,
        "Prev_R²": round(prev_r2, 5),
        "New_R²": round(new_r2, 5),
        "ΔR²": round(delta, 5),
        "Decision": (
            "Improved" if delta >= improvement_threshold
            else "Stopped" if delta < -decrease_threshold
            else "Ignored"
        )
    })

    # Decision logic
    if delta >= improvement_threshold:
        X_current = X_temp_train
        prev_r2 = new_r2
        selected_cols.append(col)
    elif delta < -decrease_threshold:
        break
# Convert logs to a DataFrame for easy review
log_df = pd.DataFrame(log_records)

print("\nFeature Selection Log:")
print(log_df)

print("\nSelected features:")
print(selected_cols)
print(f"\nFinal train R²: {prev_r2:.5f}")

In [ ]:
# Ensure the iteration column exists and is numeric

log_df = log_df.copy().reset_index().rename(columns={"index": "Iteration"})
log_df["Iteration"] = log_df["Iteration"].astype(int)

plt.figure(figsize=(18, 7))
sns.set_theme(style="whitegrid")

# Define colors for each decision outcome
decision_colors = {
    "Improved": "green",
    "Ignored": "gray",
    "Stopped": "red"
}

# Plot the R² progression line
plt.plot(
    log_df["Iteration"],
    log_df["New_R²"],
    color="steelblue",
    linewidth=2,
    marker="o",
    label="Train R²"
)

# Safely plot each decision type (skip if empty)
for decision, color in decision_colors.items():
    subset = log_df[log_df["Decision"] == decision]
    if not subset.empty:
        plt.scatter(
            subset["Iteration"],
            subset["New_R²"],
            color=color,
            s=60,
            label=decision
        )

# Annotate only features that were accepted (Improved)
for _, row in log_df[log_df["Decision"] == "Improved"].iterrows():
    plt.text(
        row["Iteration"],
        row["New_R²"] + 0.0005,  # offset to avoid overlap
        row["Feature"],
        fontsize=8,
        rotation=45,
        ha="right",
        color="green"
    )

# Titles and labels
plt.title("R² Progression During Sequential Feature Selection", fontsize=16, pad=15)
plt.xlabel("Iteration (Feature Tested)", fontsize=12)
plt.ylabel("R² (Train Set)", fontsize=12)

# Adjust x-ticks for 40+ features
plt.xticks(
    range(0, len(log_df), max(1, len(log_df)//20)),  # show every ~5%
    fontsize=9,
)
plt.yticks(fontsize=9)
plt.legend()
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
corr_matrix = X_current.corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title("Correlation matrix between features")
plt.show()

In [ ]:

lr = LinearRegression()
lr_model = lr.fit(X_train, y_train)

predictions = lr_model.predict(X_test)
train_predictions = lr_model.predict(X_train)

# Create DataFrame for plotting
df_plot = pd.DataFrame({
    'pr_result': predictions,
    'pr_error': y_test - predictions
})

# Create figure with 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- (1) Distribution plot: Actual vs Predicted ---
sns.kdeplot(y_test, color="g", label="Actual Values", ax=axes[0])
sns.kdeplot(df_plot['pr_result'], color="r", label="Predicted Values", ax=axes[0])
axes[0].set_title("Distribution: Actual vs Predicted Values", fontsize=12)
axes[0].legend()
axes[0].set_xlabel("Target Value")
axes[0].set_ylabel("Density")

# --- (2) Prediction error (residuals) scatter plot ---
sns.scatterplot(x=df_plot.index, y='pr_error', data=df_plot, color="r", ax=axes[1])
axes[1].set_title("Prediction Error (Residuals)", fontsize=12)
axes[1].set_ylabel("Prediction Error")
axes[1].set_xlabel("Sample Index")
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)  # reference line at 0
axes[1].grid(True, linestyle=':', linewidth=0.7)

plt.tight_layout()
plt.show()

print("MAE (Train):", mean_absolute_error(y_train, train_predictions))
print("MAE (Test):", mean_absolute_error(y_test, predictions))
print("R² (Train):", r2_score(y_train, train_predictions))
print("R² (Test):", r2_score(y_test, predictions))

### Polynomial

In [ ]:
optimal_columns = ['artist_song_count', 'album_freq', 'artists_avg_popularity',
       'ambient_level', 'target_class_class_65', 'target_class_class_73']

dummy_cols = ['target_class_class_65', 'target_class_class_73']

numerical_columns = ['artist_song_count', 'album_freq', 'artists_avg_popularity',
       'ambient_level']

In this part, we perform polynomial analysis for degrees ranging from 2 to 7, recording the R², MSE, and corresponding degree to identify the optimal polynomial degree.

It is worth noting that the dummy variables are separated from the numerical features in both the training and test datasets to avoid creating meaningless polynomial interactions, and are later combined with the polynomial-transformed numerical features for model training and evaluation.

In [ ]:
degree_range = range(2, 7)
degrees, mse_values, r2_values = [], [], []

X_train, X_test, y_train, y_test = train_test_split(
        X[optimal_columns], y, test_size=0.2, random_state=seed
    )


X_train_dummy = X.loc[X_train.index, dummy_cols]
X_test_dummy = X.loc[X_test.index, dummy_cols]

for n in degree_range:
    poly = PolynomialFeatures(degree=n, include_bias=False)

    # Create polynomial features for numerical columns
    X_train_pr = poly.fit_transform(X_train[numerical_columns])
    X_test_pr = poly.transform(X_test[numerical_columns])

    # Convert to DataFrames with proper names and indices
    X_train_pr = pd.DataFrame(
        X_train_pr,
        columns=poly.get_feature_names_out(numerical_columns),
        index=X_train.index
    )
    X_test_pr = pd.DataFrame(
        X_test_pr,
        columns=poly.get_feature_names_out(numerical_columns),
        index=X_test.index
    )

    # Combine polynomial features with dummy columns
    X_train_final = pd.concat([X_train_pr, X_train_dummy], axis=1)
    X_test_final = pd.concat([X_test_pr, X_test_dummy], axis=1)

    # Train and predict
    lr = LinearRegression()
    lr_model = lr.fit(X_train_final, y_train)
    y_pred = lr.predict(X_test_final)

    # Metrics for this degree
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    degrees.append(n)
    mse_values.append(mse)
    r2_values.append(r2)

# === Plot results ===
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# MSE plot
axes[0].plot(degrees, mse_values, marker='o', color='red')
axes[0].set_title('MSE vs Polynomial Degree')
axes[0].set_xlabel('Polynomial Degree')
axes[0].set_ylabel('Mean Squared Error')
axes[0].grid(True)

# R² plot
axes[1].plot(degrees, r2_values, marker='o', color='blue')
axes[1].set_title('R² vs Polynomial Degree')
axes[1].set_xlabel('Polynomial Degree')
axes[1].set_ylabel('R² Score')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:

poly = PolynomialFeatures(degree=5, include_bias=False)

X_train_pr = poly.fit_transform(X_train[numerical_columns])
X_test_pr = poly.transform(X_test[numerical_columns])

X_train_dummy = X.loc[X_train.index, dummy_cols]
X_test_dummy = X.loc[X_test.index, dummy_cols]
    # Convert to DataFrames
X_train_pr = pd.DataFrame(
    X_train_pr,
    columns=poly.get_feature_names_out(numerical_columns),
    index=X_train.index
)
X_test_pr = pd.DataFrame(
    X_test_pr,
    columns=poly.get_feature_names_out(numerical_columns),
    index=X_test.index
)

X_train_final = pd.concat([X_train_pr, X_train_dummy], axis=1)
X_test_final = pd.concat([X_test_pr, X_test_dummy], axis=1)

lr = LinearRegression()
lr_model = lr.fit(X_train_final, y_train)

predictions = lr_model.predict(X_test_final)
# Create DataFrame for plotting
df_plot = pd.DataFrame({
    'pr_result': predictions,
    'pr_error': y_test - predictions
})

# Create figure with 1 row, 2 columns
fig, axes = plt.subplots(1, 2, figsize=(14, 6))


# --- (1) Distribution plot: Actual vs Predicted ---
sns.kdeplot(y_test, color="g", label="Actual Values", ax=axes[0])
sns.kdeplot(df_plot['pr_result'], color="r", label="Predicted Values", ax=axes[0])
axes[0].set_title("Distribution: Actual vs Predicted Values", fontsize=12)
axes[0].legend()
axes[0].set_xlabel("Target Value")
axes[0].set_ylabel("Density")

# --- (2) Prediction error (residuals) scatter plot ---
sns.scatterplot(x=df_plot.index, y='pr_error', data=df_plot, color="r", ax=axes[1])
axes[1].set_title("Prediction Error (Residuals)", fontsize=12)
axes[1].set_ylabel("Prediction Error")
axes[1].set_xlabel("Sample Index")
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)  # reference line at 0
axes[1].grid(True, linestyle=':', linewidth=0.7)

plt.tight_layout()
plt.show()

print("MAE (Test):", mean_absolute_error(y_test, predictions))
print("R² (Test):", r2_score(y_test, predictions))

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X_train_final.columns,
    'Coefficient': lr_model.coef_
})

# Filter only dummy variable coefficients
dummy_coef = coef_df[coef_df['Feature'].isin(dummy_cols)]

print("Dummy Variable Coefficients:")
print(dummy_coef)

### Dummy Variable Coefficients Interpretation

| Category | Coefficient | Interpretation (vs. Baseline: `class_45`)                                     |
|-----------|-------------|-------------------------------------------------------------------------------|
| class_65  | 0.007705    | Slightly **increase** predicted value by 0.0022 units compared to `class_45`. |
| class_73  | 0.209767    | **Increases** predicted value by 0.2022 units compared to `class_45`.         |
| class_45  | — (baseline) | Reference category — effect implicitly set to 0.                              |


## Comparasion Simple Regrssion Multiple Regression


#### Simple Regression

* **Feature used:** `artists_avg_popularity`
* **R²:** 0.6294
* **MSE/MAE:** Lowest among single-feature models
* **Interpretation:**
  This single feature (`artists_avg_popularity`) explains about **62.9%** of the variability in the target variable. It performs well on its own, showing that artist popularity has a strong individual relationship with the target.

---

#### Multiple Regression

* **Selected features:**
  `artists_avg_popularity`, `album_freq`, `target_class_class_65`, `target_class_class_73`, `artist_song_count`, `ambient_level`
* **R² (test):** 0.7369
* **Interpretation:**
  By combining multiple predictors, the model explains about **74.7%** of the target’s variability — a noticeable improvement over the simple regression. This means that while `artists_avg_popularity` remains a key predictor, the additional features contribute complementary information that enhances predictive accuracy.

---

####  Overall Comparison

| Aspect                      | Simple Regression                             | Multiple Regression                                                      |
|-----------------------------| --------------------------------------------- |--------------------------------------------------------------------------|
| **Features Used**           | 1 (`artists_avg_popularity`)                  | 6 total                                                                  |
| **R² (Test)**               | 0.6294                                        | 0.7369                                                                   |
| **Error Metrics (MSE/MAE)** | Lowest among singles                          | Expected to improve further                                              |

---

#### Conclusion

The **multiple regression model** clearly outperforms the simple one.
Although `artists_avg_popularity` is the single most important predictor, including additional features such as `album_freq`, `artist_song_count`, and class indicators provides a **richer and more accurate representation** of the factors influencing the target.


# Classification

## Holdout

In [ ]:
data_set = df.copy()
y = data_set['target_class']
X = data_set.drop(columns=["target_class"])

In [ ]:
#Hold out method
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

The following reduces the features to 2D for visualization. In the PCA graph, we can observe that class_65 is far from the other classes, which could potentially result in high precision and recall. In contrast, class_45 and class_73 overlap, which can potentially lead to lower precision and recall because the models cannot separate them linearly.


In [ ]:
pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train_scaled)

# Plot
plt.figure(figsize=(8,6))
for cls, color in zip(np.unique(y_train), ['red', 'green', 'blue']):
    plt.scatter(
        X_train_2d[y_train == cls, 0],
        X_train_2d[y_train == cls, 1],
        label=f"Class {cls}",
        alpha=0.6
    )

plt.title("2D PCA Projection of Training Data")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend()
plt.grid(True)
plt.show()

### Logistic Regression

 Logistic Regression (OvR, liblinear) is trained on the scaled data, then predictions are made on the test set and evaluated using precision, recall, and F1-score.


In [ ]:

model = LogisticRegression(
    multi_class='ovr',  # one-vs-rest
    solver='liblinear', # liblinear works well for OvR
    max_iter=5000
)

# Fit the model
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

# Evaluate the model
print("\nClassification Report:\n", classification_report(y_test, y_pred))

### LDA

In [ ]:
lda = LDA()
lda.fit(X_train_scaled, y_train)
y_pred_lda = lda.predict(X_test_scaled)

print("\nLDA Classification Report:\n", classification_report(y_test, y_pred_lda))


### QDA

We test different QDA regularization values (`reg_param`) to stabilize class covariance matrices. For each value, the model is trained on the scaled training set and evaluated on the test set. Accuracy and macro F1-score are recorded to identify the regularization that gives the best overall performance.


In [ ]:

# Define a range of regularization values to test
reg_params = np.linspace(0.1, 1, 11)

results = []

for reg in reg_params:
    qda = QDA(reg_param=reg)
    qda.fit(X_train_scaled, y_train)
    y_pred_qda = qda.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred_qda)
    f1 = f1_score(y_test, y_pred_qda, average="macro")

    results.append({
        "reg_param": reg,
        "accuracy": acc,
        "macro_f1": f1
    })
# Convert results to DataFrame for analysis
results_df = pd.DataFrame(results)

# Assume results_df from the previous loop exists
plt.figure(figsize=(10, 6))

# Plot Accuracy
plt.plot(results_df["reg_param"], results_df["accuracy"], marker='o', linestyle='-', color='blue', label="Accuracy")
plt.plot(results_df["reg_param"], results_df["macro_f1"], marker='s', linestyle='--', color='green', label="Macro F1")

# Highlight the best reg_param (highest Macro F1)
best_idx = results_df["macro_f1"].idxmax()
best_reg = results_df.loc[best_idx, "reg_param"]
best_f1 = results_df.loc[best_idx, "macro_f1"]
plt.scatter(best_reg, best_f1, color='red', s=100, zorder=5, label=f"Best reg_param = {best_reg:.2f}")

# Labels and formatting
plt.title("QDA Performance vs Regularization Parameter (reg_param)", fontsize=14)
plt.xlabel("reg_param", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.xticks(results_df["reg_param"])
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()
results_df


In [ ]:
qda_best = QDA(reg_param=best_reg)
qda_best.fit(X_train_scaled, y_train)
y_pred_best = qda_best.predict(X_test_scaled)

print("\nFinal QDA Classification Report (Best reg_param):")
print(classification_report(y_test, y_pred_best))

### Compare Result

Confusion matrices are plotted for Logistic Regression, LDA, and QDA (with best regularization) to visually compare model performance. Each matrix shows the counts of true vs. predicted class labels, helping identify which classes are correctly classified and where misclassifications occur.


In [ ]:
models = {
    "Logistic Regression": y_pred,
    "LDA": y_pred_lda,
    "QDA": y_pred_best
}
plt.figure(figsize=(15, 4))
for i, (name, preds) in enumerate(models.items(), 1):
    plt.subplot(1, 3, i)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
plt.tight_layout()
plt.show()

A bar plot compares the accuracy of Logistic Regression, LDA, and QDA (with best regularization) on the test set.


In [ ]:
accuracies = {
    "Logistic Regression": accuracy_score(y_test, y_pred),
    "LDA": accuracy_score(y_test, y_pred_lda),
    "QDA": accuracy_score(y_test, y_pred_best)
}

plt.figure(figsize=(6, 4))
sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()))
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.show()

Using the hold-out method (80%-20% train-test split), we evaluated three classification models: Logistic Regression, LDA, and QDA with optimal regularization.

* **Logistic Regression** achieved the highest overall accuracy (0.80), performing particularly well for class_65 and maintaining balanced precision and recall across classes.
* **QDA** with the best `reg_param` achieved an accuracy of 0.75, benefiting from modeling class-specific covariances, which improved performance for some classes but was slightly lower than Logistic Regression overall.
* **LDA** reached 0.74 accuracy, showing consistent but slightly lower performance across all classes.

Overall, **Logistic Regression** was the best-performing classifier for this dataset.

Additionally, it is notable that **class_65 has the highest precision across all classification models**. This is expected, as in the PCA graph we can observe that class_65 is far from the other classes, resulting in high precision and recall. In contrast, **class_45 and class_73 overlap**, leading to lower precision and recall because the models cannot separate them linearly.


## Cross Validation


### Model Comparison


In [ ]:
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(solver='liblinear', C=0.05, multi_class='ovr', random_state=42))]),
    'LDA': Pipeline([('scaler', StandardScaler()), ('model', LDA())]),
    'QDA': Pipeline([('scaler', StandardScaler()), ('model', QDA(reg_param=0.1))])
}

results_per_k_fold = {}

for n in range(5, 11, 5):
    k_folds = KFold(n_splits = n, shuffle = True, random_state = 42)
    results_per_k_fold[n] = {}

    for name, model in models.items():
        scores = cross_val_score(model, X, y, scoring = 'accuracy', cv = k_folds, n_jobs = -1)
        y_pred = cross_val_predict(model, X, y, cv = k_folds)

        results_per_k_fold[n][name] = (mean(scores), np.std(scores))

        print(f"{name} - k fold: {n}")
        print(f"Scores: {scores}")
        print("Classification Report:")
        print(classification_report(y, y_pred))
        print("Accuracy: %.3f (%.3f)" % (mean(scores), np.std(scores)))
        print("\n")


Melhor modelo para k=5: LDA com acurácia de 0.947 (std: 0.105)
Melhor modelo para k=10: LDA com acurácia de 0.976 (std: 0.071)


O **melhor modelo geral é LDA com k=10**, alcançando uma **acurácia de 0.976**.

Primeiro, vamos encontrar o melhor reg_param para o QDA usando validação cruzada e plotar o desempenho.


In [ ]:
# Define a range of regularization values to test
reg_params = np.linspace(0.1, 1.0, 11)

# Store results
qda_results = []

# Define the cross-validation strategy
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for reg in reg_params:
    # Create a pipeline with scaling and QDA
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('qda', QDA(reg_param=reg))
    ])

    # Perform cross-validation
    scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')

    # Store the mean and standard deviation of the accuracy
    qda_results.append({
        "reg_param": reg,
        "mean_accuracy": scores.mean(),
        "std_accuracy": scores.std()
    })

# Convert results to a DataFrame for analysis
qda_results_df = pd.DataFrame(qda_results)

# Find the best regularization parameter
best_reg_param = qda_results_df.loc[qda_results_df['mean_accuracy'].idxmax()]
best_reg = best_reg_param['reg_param']

print(f"Best reg_param for QDA: {best_reg:.2f} with accuracy: {best_reg_param['mean_accuracy']:.3f}")

# Plot QDA Performance vs. Regularization Parameter
plt.figure(figsize=(10, 6))
plt.plot(qda_results_df["reg_param"], qda_results_df["mean_accuracy"], marker='o', linestyle='-', label="Mean Accuracy")
plt.fill_between(
    qda_results_df["reg_param"],
    qda_results_df["mean_accuracy"] - qda_results_df["std_accuracy"],
    qda_results_df["mean_accuracy"] + qda_results_df["std_accuracy"],
    alpha=0.2,
    label="Std. Dev."
)

# Highlight the best point
plt.scatter(best_reg, best_reg_param['mean_accuracy'], color='red', s=100, zorder=5, label=f"Best reg_param = {best_reg:.2f}")

plt.title("QDA Performance vs. Regularization (10-Fold CV)")
plt.xlabel("reg_param")
plt.ylabel("Accuracy")
plt.xticks(reg_params)
plt.legend()
plt.grid(True)
plt.show()

Agora, vamos comparar os três modelos usando o melhor reg_param para o QDA, gerar as matrizes de confusão e o gráfico de barras de acurácia.

In [ ]:
# Define models with the best QDA reg_param
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(solver='liblinear', multi_class='ovr', random_state=42))]),
    'LDA': Pipeline([('scaler', StandardScaler()), ('model', LDA())]),
    'QDA': Pipeline([('scaler', StandardScaler()), ('model', QDA(reg_param=best_reg))])
}

# Store predictions and scores
predictions = {}
accuracies = {}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    # Get cross-validated predictions
    y_pred = cross_val_predict(model, X, y, cv=cv)
    predictions[name] = y_pred

    # Calculate accuracy
    accuracies[name] = accuracy_score(y, y_pred)

    print(f"--- {name} ---")
    print(f"Cross-Validated Accuracy: {accuracies[name]:.3f}")
    print(classification_report(y, y_pred))
    print("\n")

# Plot Confusion Matrices
plt.figure(figsize=(18, 5))
for i, (name, y_pred) in enumerate(predictions.items(), 1):
    plt.subplot(1, 3, i)
    cm = confusion_matrix(y, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y), yticklabels=np.unique(y))
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# Plot Model Accuracy Comparison
plt.figure(figsize=(8, 6))
sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()))
plt.title("Model Accuracy Comparison (10-Fold CV)")
plt.ylabel("Accuracy")
plt.ylim(0.6, 1.0)
for index, value in enumerate(accuracies.values()):
    plt.text(index, value + 0.01, f"{value:.3f}", ha='center')
plt.show()

Using the K-Cross Validation method, we evaluated three classification models: Logistic Regression, LDA, and QDA with optimal regularization. The comparison was performed using a 10-fold stratified cross-validation (`k=10`), which yielded the best results in the initial analysis.

*   **LDA** achieved the highest overall accuracy (**0.975**), performing particularly well for class_65 and maintaining balanced precision and recall across classes.
*   **Logistic Regression** reached **0.811** accuracy, showing consistent performance but was outperformed by LDA.
*   **QDA** with the best `reg_param` achieved an accuracy of **0.640**.

Overall, **LDA with k=10 was the best-performing classifier** for this dataset under the cross-validation setup.

Additionally, it is notable that **class_65 has the highest precision across all classification models**. This is expected, as in the PCA graph we can observe that class_65 is far from the other classes. In contrast, **class_45 and class_73 overlap**, leading to lower precision and recall because the models cannot separate them linearly.

## LOOCV

In [ ]:
def evaluate_with_loo(model, X, y, scale=True, verbose=True):
    loo = LeaveOneOut()

    # Convert to numpy arrays if needed
    X_np = np.array(X)
    y_np = np.array(y)

    # Scale features if requested
    if scale:
        scaler = StandardScaler()
        X_np = scaler.fit_transform(X_np)

    y_true, y_pred = [], []

    for train_idx, test_idx in loo.split(X_np):
        X_train, X_test = X_np[train_idx], X_np[test_idx]
        y_train, y_test = y_np[train_idx], y_np[test_idx]

        model.fit(X_train, y_train)
        y_pred.append(model.predict(X_test)[0])
        y_true.append(y_test[0])

    # Compute metrics
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    report = classification_report(y_true, y_pred, output_dict=False)

    if verbose:
        print("\n" + "=" * 60)
        print(f"Model: {model.__class__.__name__}")
        print("=" * 60)
        print(report)
        print(f"Accuracy: {acc:.4f}")
        print(f"Macro F1-score: {macro_f1:.4f}")

    return {
        "model": model.__class__.__name__,
        "y_true": y_true,
        "y_pred": y_pred,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "report": report
    }

### Logistic Regression

In [ ]:
# Initialize the model
model = LogisticRegression(
    multi_class='ovr',  # one-vs-rest
    solver='liblinear', # liblinear works well for OvR
    max_iter=1000
)

logreg_result = evaluate_with_loo(model, X, y)

### LDA


In [ ]:
lda_result = evaluate_with_loo(LDA(), X, y)


### QDA


In [ ]:
# Define range of QDA regularization parameters
reg_params = np.linspace(0.1, 1.0, 5)  # 0.0, 0.1, ..., 1.0

qda_results_list = []

# Evaluate QDA for each reg_param
for reg in reg_params:
    result = evaluate_with_loo(QDA(reg_param=reg), X, y, verbose=False)
    qda_results_list.append({
        "reg_param": reg,
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"]
    })

# Convert results to DataFrame
qda_results_df = pd.DataFrame(qda_results_list)

# Find the best reg_param based on Macro F1
best_idx = qda_results_df["macro_f1"].idxmax()
best_reg = qda_results_df.loc[best_idx, "reg_param"]
best_macro_f1 = qda_results_df.loc[best_idx, "macro_f1"]

print(f"\nBest QDA reg_param: {best_reg:.2f} with Macro F1-score: {best_macro_f1:.4f}")

# ----------------------------------------------------------
# Plot Accuracy and Macro F1 vs reg_param
# ----------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.plot(qda_results_df["reg_param"], qda_results_df["accuracy"], marker='o', linestyle='-', label="Accuracy")
plt.plot(qda_results_df["reg_param"], qda_results_df["macro_f1"], marker='s', linestyle='--', label="Macro F1")

# Highlight best reg_param
plt.scatter(best_reg, best_macro_f1, color='red', s=100, zorder=5, label=f"Best reg_param = {best_reg:.2f}")

plt.title("QDA Performance vs Regularization Parameter (LOOCV)")
plt.xlabel("reg_param")
plt.ylabel("Score")
plt.xticks(reg_params)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

In [ ]:
print("\nClassification report for best QDA reg_param:")
qda_best_result = evaluate_with_loo(QDA(reg_param=best_reg), X, y, verbose=True)

In [ ]:
models_preds = {
    "Logistic Regression": logreg_result,
    "LDA": lda_result,
    "QDA (best reg_param)": qda_best_result
}

# ----------------------------------------------------------
# Plot confusion matrices
# ----------------------------------------------------------
plt.figure(figsize=(15, 4))
for i, (name, result) in enumerate(models_preds.items(), 1):
    plt.subplot(1, 3, i)
    cm = confusion_matrix(result["y_true"], result["y_pred"])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
accuracies = {
    "Logistic Regression": logreg_result["accuracy"],
    "LDA": lda_result["accuracy"],
    "QDA (best reg_param)": qda_best_result["accuracy"]
}

# Plot bar chart
plt.figure(figsize=(6, 4))
sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()))
plt.title("Model Accuracy Comparison (LOOCV)")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## Model Performance Analysis

**LDA**

* Assumes each class is Gaussian with the **same covariance**.
* `class_65` is well-separated; `class_45` and `class_73` slightly overlap.
* LDA can find a linear combination that perfectly separates classes.
* **Accuracy:** 1.0

**Logistic Regression (OvR)**

* Assumes **linear decision boundaries** per class.
* Overlap between `class_45` and `class_73` causes misclassifications.
* **Accuracy:** ~0.82

**QDA**

* Assumes each class has its **own covariance** → flexible boundaries.
* Covariance estimation can be noisy with 1000 samples per class, even with regularization.
* Misclassifications occur mostly between overlapping classes.
* **Accuracy:** ~0.74


## Bootstrap

In [ ]:
# ===============================================================
# Load and preprocess data
# ===============================================================

data_set = df.copy()
y = data_set['target_class']
X = data_set.drop(columns=["target_class"])

# ===============================================================
# Bootstrap Evaluation Function
# ===============================================================
def evaluate_with_bootstrap(model, X, y, n_bootstrap=100, scale=True, verbose=True):
    X_np = np.array(X)
    y_np = np.array(y)

    if scale:
        scaler = StandardScaler()
        X_np = scaler.fit_transform(X_np)

    n_samples = len(y_np)
    acc_scores, f1_scores = [], []

    for b in range(n_bootstrap):
        # Sample with replacement
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        oob_indices = np.setdiff1d(np.arange(n_samples), indices)

        if len(oob_indices) == 0:
            continue  # skip if no OOB samples

        X_train, y_train = X_np[indices], y_np[indices]
        X_test, y_test = X_np[oob_indices], y_np[oob_indices]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc_scores.append(accuracy_score(y_test, y_pred))
        f1_scores.append(f1_score(y_test, y_pred, average="macro"))

    acc_mean, acc_std = np.mean(acc_scores), np.std(acc_scores)
    f1_mean, f1_std = np.mean(f1_scores), np.std(f1_scores)

    if verbose:
        print("\n" + "=" * 60)
        print(f"Model: {model.__class__.__name__} (Bootstrap Evaluation)")
        print("=" * 60)
        print(f"Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
        print(f"Macro F1-score: {f1_mean:.4f} ± {f1_std:.4f}")

    return {
        "model": model.__class__.__name__,
        "accuracy": acc_mean,
        "accuracy_std": acc_std,
        "macro_f1": f1_mean,
        "macro_f1_std": f1_std
    }

# ===============================================================
# Logistic Regression
# ===============================================================
logreg_result = evaluate_with_bootstrap(
    LogisticRegression(multi_class='ovr', solver='liblinear', max_iter=1000),
    X, y
)

# ===============================================================
# LDA
# ===============================================================
lda_result = evaluate_with_bootstrap(LDA(), X, y)

# ===============================================================
# QDA with regularization search
# ===============================================================
reg_params = np.linspace(0.1, 1.0, 5)
qda_results_list = []

for reg in reg_params:
    result = evaluate_with_bootstrap(QDA(reg_param=reg), X, y, verbose=False)
    qda_results_list.append({
        "reg_param": reg,
        "accuracy": result["accuracy"],
        "macro_f1": result["macro_f1"]
    })

qda_results_df = pd.DataFrame(qda_results_list)
best_idx = qda_results_df["macro_f1"].idxmax()
best_reg = qda_results_df.loc[best_idx, "reg_param"]
best_macro_f1 = qda_results_df.loc[best_idx, "macro_f1"]

print(f"\nBest QDA reg_param: {best_reg:.2f} with Macro F1-score: {best_macro_f1:.4f}")

# ----------------------------------------------------------
# Plot QDA performance vs reg_param
# ----------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.plot(qda_results_df["reg_param"], qda_results_df["accuracy"], marker='o', label="Accuracy")
plt.plot(qda_results_df["reg_param"], qda_results_df["macro_f1"], marker='s', linestyle='--', label="Macro F1")
plt.scatter(best_reg, best_macro_f1, color='red', s=100, label=f"Best reg_param = {best_reg:.2f}")
plt.title("QDA Performance vs Regularization Parameter (Bootstrap)")
plt.xlabel("reg_param")
plt.ylabel("Score")
plt.xticks(reg_params)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

# ===============================================================
# Best QDA final evaluation
# ===============================================================
qda_best_result = evaluate_with_bootstrap(QDA(reg_param=best_reg), X, y)

# ===============================================================
# Compare model accuracies
# ===============================================================
accuracies = {
    "Logistic Regression": logreg_result["accuracy"],
    "LDA": lda_result["accuracy"],
    "QDA (best reg_param)": qda_best_result["accuracy"]
}

plt.figure(figsize=(6, 4))
sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()))
plt.title("Model Accuracy Comparison (Bootstrap)")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()



Bootstrap Evaluation Analysis

The bootstrap evaluation provides an estimate of each model’s **stability and generalization** by resampling the dataset multiple times.

* **LDA** achieved the highest mean accuracy (**0.8942 ± 0.1272**), confirming its strong performance when data roughly follows its assumptions. However, the relatively high standard deviation indicates that its performance can fluctuate depending on the sampled data — suggesting some sensitivity to class balance or overlap.
* **Logistic Regression** was slightly less accurate (**0.8090 ± 0.0117**) but much more stable (low variance), showing consistent results across bootstrap samples. This indicates good robustness, even if its linear boundaries limit peak performance.
* **QDA** again performed the weakest (**0.7340 ± 0.0110**), with modest variability. Its flexibility in modeling class-specific covariances did not pay off here, likely due to noisy covariance estimation.

---

Conclusion

The bootstrap results reinforce the earlier findings:
**LDA generalizes best overall**, benefiting from well-matched assumptions about the data, while **Logistic Regression** provides a stable and reliable baseline. **QDA**, though more flexible, underperforms due to overfitting risks and unstable parameter estimates.



# Feature Selection

helps to understand how strongly each feature is related to the target class —  measures how well a feature separates the classes.

In [ ]:

# Identify constant features
constant_features = X.columns[X.nunique() <= 1].tolist()
print("Constant features (will be skipped):", constant_features)

# Select only non-constant features
X_nonconstant = X.drop(columns=constant_features)

# Compute F-values and p-values
F_values, p_values = f_classif(X_nonconstant, y)

# Create DataFrame for all features
feature_stats = pd.DataFrame({
    'Feature': X_nonconstant.columns,
    'F_value': F_values,
    'p_value': p_values
}).sort_values(by='F_value', ascending=False)

# Print all values
pd.set_option('display.max_rows', None)
print(feature_stats)

In [ ]:
data_set = df.copy()
y = data_set['target_class']
X = data_set.drop(columns=["target_class"])

def ridge_classification(train_x, train_y, test_x, test_y, alpha):
    ridgereg  = LogisticRegression(
        penalty='l2',       # Ridge
        C=1/alpha,            # inverse of regularization strength
        multi_class='ovr',  # One-vs-Rest
        max_iter=5000
    )
    ridgereg.fit(train_x, train_y)

    train_y_pred = ridgereg.predict(train_x)
    test_y_pred = ridgereg.predict(test_x)

    # Accuracy metrics
    accuracy_train = (train_y_pred == train_y).mean()
    accuracy_test = (test_y_pred == test_y).mean()

    # Use only first class coefficients
    coef_flat = ridgereg.coef_[0]
    return [accuracy_train, accuracy_test, *coef_flat]


def lasso_classification(train_x, train_y, test_x, test_y, alpha):
    if alpha == 0:
        model = LogisticRegression(
            penalty=None,
            solver='saga',
            multi_class='ovr',
            max_iter=10000
        )
    else:
        C = 1 / alpha
        model = LogisticRegression(
            penalty='l1',
            C=C,
            solver='saga',
            multi_class='ovr',
            max_iter=10000
        )

    model.fit(train_x, train_y)

    # Predictions
    train_y_pred = model.predict(train_x)
    test_y_pred = model.predict(test_x)

    # Accuracy
    accuracy_train = (train_y_pred == train_y).mean()
    accuracy_test = (test_y_pred == test_y).mean()

    # Take only first class coefficients to match feature count
    coef_flat = model.coef_[0]  # shape = (n_features,)

    return [accuracy_train, accuracy_test, *coef_flat]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit on train only
X_test_scaled = scaler.transform(X_test)

### Ridge Regression

In [ ]:
alpha_ridge = np.logspace(-6, 3, 20)

# DataFrame setup
feature_names = X.columns
col = ['accuracy_train', 'accuracy_test'] + list(feature_names)
ind = [f'alpha_{alpha:.2g}' for alpha in alpha_ridge]
coef_matrix_ridge = pd.DataFrame(index=ind, columns=col, dtype=float)

# Loop
for i, alpha in enumerate(alpha_ridge):
    results = ridge_classification(
        X_train_scaled, y_train, X_test_scaled, y_test, alpha
    )
    coef_matrix_ridge.iloc[i, :] = results

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(alpha_ridge, coef_matrix_ridge['accuracy_train'], marker='o', label='Train Accuracy')
plt.plot(alpha_ridge, coef_matrix_ridge['accuracy_test'], marker='s', label='Test Accuracy')
plt.xscale('log')
plt.xlabel('Alpha')
plt.ylabel('Accuracy')
plt.title('Ridge Classification: Accuracy vs Alpha (Linear Scale)')
plt.legend()
plt.grid(True)
plt.show()

coef_matrix_ridge['alpha'] = alpha_ridge
best_idx = coef_matrix_ridge['accuracy_test'].astype(float).idxmax()

# Extract values
best_alpha = coef_matrix_ridge.loc[best_idx, 'alpha']
best_accuracy = coef_matrix_ridge.loc[best_idx, 'accuracy_test']

In [ ]:
feature_cols = feature_names.tolist()
coef_values = coef_matrix_ridge.loc[best_idx, feature_cols].astype(float)

print(f"Best alpha: {best_alpha}")
print(f"Best test accuracy: {best_accuracy:.4f}")
print("\nCoefficients for best alpha:")
for f, c in zip(feature_cols, coef_values):
    print(f"{f}: {c:.5f}")

#### **High coefficients for strongly predictive features**

* **Features:**

  * `intensity_level (-0.63835)`
  * `album_freq (-0.49043)`
  * `loudness_yeo (-1.21511)`
  * `signal_strength / signal_power (-0.55321)`
  * `positivity_index (0.23909)`
* **Reason:**

  * These features are highly predictive, as reflected by **large absolute coefficients**.
  * Many are **highly correlated with other features**:

    * `intensity_level` ↔ `loudness_yeo (r ≈ 0.96)`
    * `signal_strength` ↔ `signal_power (r = 1.0)`
    * `activity_rate` ↔ `temp_zscore (r = 1.0)`
  * Ridge shrinks correlated features together rather than eliminating them, so both get substantial coefficients.

#### **Moderate coefficients for moderately predictive features**

* **Features:**

  * `key_mode (-0.14479)`, `verbal_density (-0.10928)`, `happy_dance (-0.26015)`, `artists_avg_popularity (0.26290)`
* **Reason:**

  * These features contribute **independent or partially redundant information**.
  * Their F-values and correlations are moderate (e.g., `positivity_index ↔ happy_dance r ≈ 0.93`), so Ridge retains them with smaller but meaningful coefficients.

#### **Small coefficients for weak or redundant features**

* **Features:**

  * `time_signature (0.00202)`, `tempo_class (0.01976)`, `distorted_movement (-0.01213)`, `is_dance_hit (0.0)`, `echo_constant (0.0)`
* **Reason:**

  * Low F-values indicate weak predictive power.
  * Some are **highly redundant or constant**, so Ridge reduces their magnitude.
  * Unlike Lasso, Ridge does not make them exactly zero, but the impact is negligible.

#### **Features with extreme values due to correlation**

* **Examples:**

  * `activity_rate (65.29)` and `temp_zscore (65.29)` → perfectly correlated (r = 1.0)
  * `tempo_vs_genre (-126.22)` → highly correlated with `activity_rate` and `temp_zscore` (r ≈ 0.97)
* **Reason:** Ridge coefficients can inflate for highly correlated features to distribute the effect across them, while controlling overfitting via L2 penalty.


### Lasso

In [ ]:
# --- Set alphas ---
alpha_lasso = np.logspace(-6, 1, 10)

# --- Prepare DataFrame ---
feature_names = X.columns
col = ['accuracy_train', 'accuracy_test'] + list(feature_names)
ind = [f'alpha_{a:.2g}' for a in alpha_lasso]
coef_matrix_lasso = pd.DataFrame(index=ind, columns=col, dtype=float)

# --- Fill DataFrame with accuracies + coefficients ---
for i, alpha in enumerate(alpha_lasso):
    result = lasso_classification(
        X_train_scaled, y_train, X_test_scaled, y_test, alpha
    )
    coef_matrix_lasso.iloc[i, 0] = result[0]  # train accuracy
    coef_matrix_lasso.iloc[i, 1] = result[1]  # test accuracy
    coef_matrix_lasso.iloc[i, 2:] = result[2:]  # coefficients for all features

# --- Find best alpha ---
best_idx = coef_matrix_lasso['accuracy_test'].astype(float).idxmax()
best_alpha = alpha_lasso[ind.index(best_idx)]
best_accuracy = coef_matrix_lasso.loc[best_idx, 'accuracy_test']
print(f"Best alpha: {best_alpha:.2e}")
print(f"Best test accuracy: {best_accuracy:.4f}")

In [ ]:
# --- Extract only coefficients ---
coef_only_lasso = coef_matrix_lasso.drop(columns=['accuracy_train', 'accuracy_test'])

# --- Plot accuracy vs alpha ---
plt.figure(figsize=(8,5))
plt.plot(alpha_lasso, coef_matrix_lasso['accuracy_train'], marker='o', label='Train Accuracy')
plt.plot(alpha_lasso, coef_matrix_lasso['accuracy_test'], marker='s', label='Test Accuracy')
plt.xscale('log')
plt.xlabel('Alpha')
plt.ylabel('Accuracy')
plt.title('L1 Logistic Regression: Accuracy vs Alpha')
plt.grid(True)
plt.legend()
plt.show()

# --- Display coefficients for best alpha ---
best_coef_table = coef_only_lasso.loc[best_idx]
best_coef_table


#### High coefficients for strongly predictive features

- **Features:** `album_freq (-0.593)`, `intensity_level (-0.226)`, `duration_5 (-0.217)`, `duration_1 (-0.204)`
- **Reason:** These features had **high ANOVA F-values** (e.g., `intensity_level F ≈ 1280`) and are highly correlated with the target or other strong features.
  - Example: `intensity_level` is strongly correlated with `loudness_yeo (r ≈ 0.96)`, giving it robust predictive power.

#### Moderate coefficients for moderately predictive features

- **Features:** `key_mode (-0.135)`, `movement_index (-0.093)`, `verbal_density (-0.111)`
- **Reason:** These features contribute **unique information** that is less correlated with other strong features, so Lasso retains them with smaller coefficients.
  - Example: `key_mode` has moderate correlation with the target and low correlation with dominant features.

#### Near-zero coefficients for weak or redundant features

- **Features:** `time_signature (0.00177)`, `tempo_class (0.0438)`, `popularity_level (0.0613)`
- **Reason:**
  1. Some features are **weak predictors** (low F-values, e.g., `time_signature F ≈ 16`).
  2. Others are **highly correlated with stronger features** and thus penalized by Lasso.
     - Example: `signal_strength` and `signal_power` are perfectly correlated (r = 1.0); if one dominates, the other is shrunk.
     - Example: `activity_rate` and `temp_zscore` are perfectly correlated, so only one receives a non-zero coefficient.


### Elastic Net Regression

In [ ]:
# Suprimir avisos para consistência
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- 3. Elastic Net Regression ---

def elastic_net_classification(train_x, train_y, test_x, test_y, C_val, l1_ratio_val):
    """
    Treina um modelo de regressão logística Elastic Net.
    C_val é o inverso da força de regularização.
    l1_ratio_val é o parâmetro de mistura (0=L2, 1=L1).
    """

    if l1_ratio_val == 0:
        penalty = 'l2'
    elif l1_ratio_val == 1:
        penalty = 'l1'
    else:
        penalty = 'elasticnet'

    model = LogisticRegression(
        penalty=penalty,
        C=C_val,
        l1_ratio=l1_ratio_val if penalty == 'elasticnet' else None,
        solver='saga',
        multi_class='ovr',
        max_iter=5000  # Aumentado max_iter para o solver 'saga'
    )

    model.fit(train_x, train_y)

    # Previsões
    train_y_pred = model.predict(train_x)
    test_y_pred = model.predict(test_x)

    # Acurácia
    accuracy_train = accuracy_score(train_y, train_y_pred)
    accuracy_test = accuracy_score(test_y, test_y_pred)

    # Pega os coeficientes da primeira classe
    coef_flat = model.coef_[0]

    return [accuracy_train, accuracy_test, *coef_flat]

# --- Grid Search para Elastic Net ---

# Define a grade de parâmetros
# Usando valores C similares ao Ridge (inverso da força de regularização)
C_values = np.logspace(-6, 3, 10)
l1_ratios = np.array([0.1, 0.3, 0.5, 0.7, 0.9])

# Setup do DataFrame
col = ['C', 'l1_ratio', 'accuracy_train', 'accuracy_test']
results_list = []

print("Iniciando o Grid Search para Elastic Net (pode levar alguns minutos)...")

# Loop sobre a grade
for C_val in C_values:
    for l1_ratio_val in l1_ratios:
        # As variáveis X_train_scaled, y_train, etc. já existem da célula 5
        results = elastic_net_classification(
            X_train_scaled, y_train, X_test_scaled, y_test, C_val, l1_ratio_val
        )
        results_list.append([C_val, l1_ratio_val] + results)

print("Grid Search concluído.")

# Cria um DataFrame para os resultados
coef_cols = list(feature_names) # feature_names já existe da célula 3
all_cols = ['C', 'l1_ratio', 'accuracy_train', 'accuracy_test'] + coef_cols

elastic_net_results = pd.DataFrame(results_list, columns=all_cols)

# --- Encontra os melhores parâmetros ---
best_idx = elastic_net_results['accuracy_test'].astype(float).idxmax()
best_params_row = elastic_net_results.loc[best_idx]

best_C = best_params_row['C']
best_l1_ratio = best_params_row['l1_ratio']
best_accuracy = best_params_row['accuracy_test']

print("\n--- Resultados da Regressão Elastic Net ---")
print(f"Melhor C (inverso da regularização): {best_C:.2e}")
print(f"Melhor l1_ratio: {best_l1_ratio:.2f}")
print(f"Melhor acurácia de teste: {best_accuracy:.4f}\n")

# --- Mostra os coeficientes do melhor modelo ---
print("Coeficientes do melhor modelo Elastic Net:")
best_coefs = best_params_row[coef_cols].astype(float)

# Filtra coeficientes não-zero
non_zero_coefs = best_coefs[best_coefs.abs() > 1e-10]

print(f"\nTotal de features: {len(coef_cols)}")
print(f"Features selecionadas (coeficientes não-zero): {len(non_zero_coefs)}\n")

# Ordena por valor absoluto para ver as features mais importantes
sorted_coefs = non_zero_coefs.abs().sort_values(ascending=False).index

for f in sorted_coefs:
    print(f"{f}: {best_coefs[f]:.5f}")

# --- Plot ---
plt.figure(figsize=(10, 6))
for l1_ratio in l1_ratios:
    subset = elastic_net_results[elastic_net_results['l1_ratio'] == l1_ratio]
    plt.plot(subset['C'], subset['accuracy_test'], marker='o', label=f'l1_ratio = {l1_ratio}')

plt.xscale('log')
plt.xlabel('C (Inverso da Força de Regularização)')
plt.ylabel('Acurácia no Teste')
plt.title('Elastic Net: Acurácia vs. C para diferentes l1_ratios')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Ridge vs. Lasso vs. Elastic Net


| Model           | Most Acurate (Test) | Features Selected | Best Parameters                      |
|:----------------|:--------------------|:------------------|:-------------------------------------|
| **Ridge (L2)**  | **0.9917** 🏆       | 48 de 48          | `alpha` = 112.88 ($C \approx 0.009$) |
| **Lasso (L1)**  | 0.9650              | 44 de 48          | `alpha` = 0.046 ($C \approx 21.5$)   |
| **Elastic Net** | 0.9633              | 46 de 48          | `C` = 100.0, `l1_ratio` = 0.90       |

### Performance (Accuracy)

**Ridge Regression (L2)** was the clear winner in terms of predictive accuracy, reaching **99.17%**.

Lasso (L1) and Elastic Net performed almost identically (96.50% and 96.33%, respectively), but both were significantly worse than Ridge.

This suggests that, for our dataset, **feature selection (removing features) might not be the best strategy**. The superior performance of Ridge indicates that *all* features, even highly correlated ones, contain useful information for the model. The best approach was to keep all features and simply "shrink" their coefficients to handle multicollinearity, which Ridge does effectively.

### Coefficient Analysis and Feature Selection

* **Ridge (L2):** Kept all 48 features.It handled perfectly correlated features (like `activity_rate` and `temp_zscore`) by giving both large and identical coefficients (`65.29`). The L2 penalty successfully controlled overfitting while keeping all features.

* **Lasso (L1):** Was the most aggressive, eliminating 4 features in total (`is_dance_hit`, `echo_constant`, `duration_log`, and `duration_log_z`). This resulted in a somewhat simpler (more interpretable) model, but at a cost in accuracy.

* **Elastic Net (L1 + L2):** The best model was found with **`l1_ratio = 0.90`**.
    * This means the ideal model was **90% Lasso and 10% Ridge**.
    * It strongly wanted to perform feature selection (like Lasso), but still needed a small Ridge penalty (the 10%) to stabilize and handle correlated features.
    * This is evident in the feature count: it eliminated only 2 features (`is_dance_hit` and `echo_constant`), while pure Lasso removed 4. The small amount of L2 prevented it from removing `duration_log` and `duration_log_z`.

### Final Conclusion

Based on this analysis, two main conclusions can be drawn depending on the objective:

1. **For MAXIMUM predictive ACCURACY:** **Ridge Regression (L2)** is the undisputed choice. Our dataset benefits more from keeping all features and simply managing multicollinearity by **shrinking coefficients rather than removing them.**

2. **For a SIMPLER MODEL (Interpretability):** **Lasso Regression (L1)** would be the better choice. It sacrificed about 2.7% in accuracy, but delivered a "cleaner" model with 4 fewer features, simplifying interpretation of which features are truly important.

In this specific scenario, **Elastic Net did not provide the best result**. Its performance was slightly worse than Lasso and much worse than Ridge. However, it did a great job diagnosing the problem: its optimal parameters (`l1_ratio=0.9`) told us the problem benefits *primarily* from L1 feature selection, but that correlation among features was still a factor (hence the need for 10% L2).

*Can classification models obtain better results if they use just a few features instead of all available
features?*

For this problem the features Lasso removed contained valuable information for the classification. Shrinking the coeficients of all the features (Ridge) was more effective than eliminating them. In **this case**, with this dataset, the initial hypothesis was refuted since it showed that feature selection not only did not help but also harmed the final accuracy.

### GAM


In [ ]:
categorical_features = [
    'duration_1', 'duration_2', 'duration_3', 'duration_4', 'duration_5','loudness_level','popularity_level','tempo_class','explicit','mode_indicator','time_signature_class_boolean','is_instrumental'
]
constant_features = {'is_dance_hit','echo_constant'}

numerical_features = [
    'time_signature','key_mode','artist_song_count','album_freq','movement_index','intensity_level','verbal_density','purity_score','positivity_index','activity_rate','loudness_intensity',
    'happy_dance',
    'acoustics_instrumental',
    'artists_avg_popularity',
    'tempo_vs_genre',
    'energy_rank_pct',
    'loud_energy_ratio',
    'mood_pca',
    'mood_cluster',
    'acoustic_valence_mood_cluster',
    'signal_strength',
    'focus_factor',
    'ambient_level',
    'key_sin',
    'key_cos',
    'duration_log',
    'duration_log_z',
    'loudness_yeo',
    'temp_zscore',
    'resonance_factor',
    'timbre_index',
    'distorted_movement',
    'signal_power',
    'target_regression'
]

In [ ]:
data_set = df.copy()
y = data_set['target_class']
#Drop Constant features
X = data_set.drop(columns=["target_class"] + list(constant_features))

# Get feature indices
categorical_indices = [X.columns.get_loc(col) for col in categorical_features if col in X.columns]
numerical_indices = [X.columns.get_loc(col) for col in numerical_features if col in X.columns]

In [ ]:
from pygam import LogisticGAM, s, f
from sklearn.preprocessing import LabelEncoder

X_array = X.values

# Sort indices to maintain column order
numerical_indices_sorted = sorted(numerical_indices)
categorical_indices_sorted = sorted(categorical_indices)

# Combine and sort all indices to build terms in column order
all_indices = [(idx, 'numerical') for idx in numerical_indices] + \
              [(idx, 'categorical') for idx in categorical_indices]
all_indices.sort(key=lambda x: x[0])  # Sort by column index

# Build GAM terms in column order
terms = None
for idx, idx_type in all_indices:
    if idx_type == 'numerical':
        term = s(idx,10)
    else:  # categorical
        term = f(idx)

    if terms is None:
        terms = term
    else:
        terms += term

print(f"\nTerms built successfully in column order:")
print(f"  {len(numerical_indices)} spline terms + {len(categorical_indices)} factor terms")
print(f"  Total terms: {len(all_indices)}")

# Encode target variable
le = LabelEncoder()
y_encoded = le.fit_transform(y)
n_classes = len(le.classes_)
print(f"\nTarget classes: {le.classes_}")
print(f"Number of classes: {n_classes}")
print(f"Class distribution:\n{pd.Series(y).value_counts().sort_index()}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale all features using StandardScaler (mean 0, variance 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_array)

# List to store each one-vs-rest GAM model
models = []

# Regularization values for GAM grid search (lambda range 10 → 10,000)
lam = np.logspace(0, 4, 10)

# Train one LogisticGAM per class (one-vs-rest classification)
for class_idx in range(n_classes):
    class_name = le.classes_[class_idx]
    print(f"\nTraining model for class {class_name}...")

    # Convert multiclass target into binary (1 for current class, 0 otherwise)
    y_binary = (y_encoded == class_idx).astype(int)
    print(f"  Positive samples: {y_binary.sum()} / {len(y_binary)} ({100*y_binary.mean():.1f}%)")

    # Build a Logistic GAM model using the previously defined terms
    gam = LogisticGAM(terms, max_iter=5000, verbose=False)

    # Hyperparameter tuning using grid search on lambda values
    gam.gridsearch(X_scaled, y_binary, lam=lam)
    models.append(gam)

    train_acc = gam.accuracy(X_scaled, y_binary)
    print(f"    Training accuracy: {train_acc:.4f}")

print("\nAll models trained successfully!")

Results show that the GAM models for class_45 and class_65 achieve extremely high performance, with training accuracies of 0.9997 and 1.0000, indicating that these two classes are very easily separable in the feature space.

The model for class_45 required significantly more time, suggesting a more complex optimization landscape. In contrast, the model for class_73 performs noticeably worse, reaching a training accuracy of 0.8390, which implies that this class overlaps more with others or has more complex, nonlinear boundaries

In [ ]:
# Build the full list of feature names (numerical + categorical)
feature_names = numerical_features + categorical_features

# Map term index to feature index in X for categorical features
term_indices = [i for i, term in enumerate(models[0].terms)
                if not term.isintercept and term.feature in range(len(feature_names))]

# Number of GAM models (one per class)
n_classes = len(models)
cols = n_classes
rows = len(term_indices)

plt.figure(figsize=(6*cols, 3*rows))


# Iterate over each categorical term
for row_idx, i in enumerate(term_indices):
    # Map GAM term → original feature index in X
    feature_idx = models[0].terms[i].feature  # index of this feature in X
    term_name = feature_names[feature_idx]    # get categorical feature name

    # Plot one partial dependence curve per class (one-vs-rest models)
    for class_idx in range(n_classes):
        gam = models[class_idx]
        plt.subplot(rows, cols, row_idx*cols + class_idx + 1)

        # Generate X grid for the feature
        XX = gam.generate_X_grid(term=i)
        partial_dep, conf = gam.partial_dependence(term=i, X=XX, width=0.95)

        plt.plot(XX[:, feature_idx], partial_dep, label='Partial dependence')
        plt.plot(XX[:, feature_idx], conf, c='r', ls='--', label='95% CI')

        # Titles and labels
        plt.title(f"{term_name} - {le.classes_[class_idx]}")
        plt.xlabel(term_name)
        plt.ylabel('Effect')
        plt.tight_layout()

plt.show()

The red dashed line in the plots represents the 95% confidence interval for the estimated partial dependence. It shows the range within which the true effect of the feature is likely to lie. For most cases, this interval is fairly tight, particularly for the stable classes like class_65 and class_45, indicating that the model is confident in its estimated effect.


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay, ConfusionMatrixDisplay, precision_score, \
    recall_score, roc_auc_score

#Number of folds for cross-validation
n_splits = 5
#Number of target classes
n_classes = len(le.classes_)
#StratifiedKFold
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# ------------------------------------------------------------
# 3. Cross-validation loop
# ------------------------------------------------------------

#List to store all the metrics
all_metrics = []

for class_idx in range(n_classes):

    # Current class name (for OvR: One-vs-Rest)
    class_name = le.classes_[class_idx]
    print(f"\nOvR cross-val for class  {class_name}")

    # Convert multiclass labels to binary labels for the current class
    y_binary = (y_encoded == class_idx).astype(int)
    print(f"   positives: {y_binary.sum()} / {len(y_binary)}  "
          f"({100*y_binary.mean():.1f}%)")

    # Perform stratified cross-validation for this class
    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X_scaled, y_binary)):
        X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
        y_train, y_test = y_binary[train_idx], y_binary[test_idx]

        # fit GAM
        gam = LogisticGAM(terms, max_iter=5000, verbose=False)

        # Hyperparameter tuning for lambda smoothing parameters
        gam.gridsearch(X_train, y_train, lam=lam)

        # predict
        y_test_pred = gam.predict(X_test)        # predicted labels (0/1)
        y_test_proba = gam.predict_proba(X_test) # predicted probabilities

        # store everything
        all_metrics.append({
            'class'     : class_name,
            'fold'      : fold_idx + 1,
            'accuracy'  : accuracy_score(y_test, y_test_pred),
            'precision' : precision_score(y_test, y_test_pred, zero_division=0),
            'recall'    : recall_score(y_test, y_test_pred, zero_division=0),
            'f1'        : f1_score(y_test, y_test_pred, zero_division=0),
            'roc_auc'   : roc_auc_score(y_test, y_test_proba),
            'y_true'    : y_test,
            'y_pred'    : y_test_pred,
            'y_proba'   : y_test_proba
        })

# ------------------------------------------------------------
# 4. DataFrame for quick inspection
# ------------------------------------------------------------
metrics_df = pd.DataFrame([{k:v for k,v in m.items()
                            if k not in {'y_true','y_pred','y_proba'}}
                           for m in all_metrics])
print("\nMean test metrics per class")

# Group by class and compute average metrics across the CV folds
print(metrics_df.groupby('class')[['accuracy','precision','recall','f1','roc_auc']].mean())

The results show that class_45 and class_65 are easy for the model to separate, while class_73 is more challenging, reflected by its lower precision, recall, and F1.


In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(6,6))
for cls in le.classes_:
    # aggregate ground-truth and probabilities for this class
    y_true_all = np.hstack([m['y_true'] for m in all_metrics if m['class']==cls])
    y_proba_all = np.hstack([m['y_proba'] for m in all_metrics if m['class']==cls])

    # compute ROC
    fpr, tpr, _ = roc_curve(y_true_all, y_proba_all)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f'{cls} (AUC = {roc_auc:0.3f})')

plt.plot([0,1],[0,1],'k--',lw=.5)
plt.xlabel('False-positive rate')
plt.ylabel('True-positive rate')
plt.title('ROC curves – OvR GAM (macro-average over folds)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Blues")

# -------------------------------------------------
# 1. Build aggregated confusion matrices (OvR)
#    For each class, combine predictions from all 5 folds
# -------------------------------------------------
cms, labels = [], []
global_max = 0  # to normalize color scale across all heatmaps
for cls in le.classes_:
    # Stack all true and predicted labels from CV for this class
    y_true_all = np.hstack([m['y_true'] for m in all_metrics if m['class']==cls])
    y_pred_all = np.hstack([m['y_pred'] for m in all_metrics if m['class']==cls])

    # Compute binary confusion matrix for this OvR class
    cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
    cms.append(cm)
    labels.append(cls)

    # Track max value to keep heatmap scales consistent
    global_max = max(global_max, cm.max())

# -------------------------------------------------
# 2.  plot
# -------------------------------------------------
fig, axes = plt.subplots(1, n_classes, figsize=(4*n_classes, 4), sharey=False)
for ax, cm, cls in zip(axes, cms, labels):
    sns.heatmap(cm, annot=False, fmt="d", cmap="Blues",
                cbar=ax is axes[-1], vmin=0, vmax=global_max,
                xticklabels=['other', cls],
                yticklabels=['other', cls],
                ax=ax, linewidths=.5)

    # annotate cells: count + percentage
    total = cm.sum()
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            count = cm[i, j]
            pct = count / total * 100
            ax.text(j+.5, i+.5, f'{count}\n({pct:.1f}%)',
                    ha="center", va="center",
                    color="white" if count > global_max/2 else "black",
                    fontsize=11, weight='bold')

    ax.set_title(f'Confusion Matrix – {cls}', fontsize=13, weight='bold')
    ax.set_xlabel('Predicted', fontsize=11)
    if ax is axes[0]:
        ax.set_ylabel('Actual', fontsize=11)

plt.suptitle('OvR-GAM: aggregated over 5-fold CV', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

plt.figure(figsize=(6,5))
for cls in le.classes_:
    y_true_all = np.hstack([m['y_true'] for m in all_metrics if m['class']==cls])
    y_proba_all = np.hstack([m['y_proba'] for m in all_metrics if m['class']==cls])
    precision, recall, _ = precision_recall_curve(y_true_all, y_proba_all)
    ap = average_precision_score(y_true_all, y_proba_all)
    plt.plot(recall, precision, label=f'{cls}  (AP={ap:.3f})')

plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('PR curves – OvR GAM (macro-average over folds)')
plt.legend()
plt.show()

The PR curves show that the model achieves high precision and recall for classes like class_45 and class_65, with AP scores near 1, while class_73 is more challenging, reflected in a lower curve and AP around 0.71.


In [ ]:
# For each class in the label encoder
for cls in le.classes_:

    # Collect all predicted probabilities (OvR) across all CV folds
    prob_pos = np.hstack([m['y_proba'] for m in all_metrics if m['class']==cls])

    # Corresponding ground-truth binary labels (0 = other, 1 = target class)
    y_true   = np.hstack([m['y_true'] for m in all_metrics if m['class']==cls])

    # Split predicted probabilities by true class
    prob_neg = prob_pos[y_true == 0]
    prob_pos = prob_pos[y_true == 1]

    # -----------------------------
    # Plot probability distributions
    # -----------------------------
    plt.figure(figsize=(5,3))

    # Distribution for negative class (should be near 0 ideally)
    sns.histplot(prob_neg, bins=30, kde=True, color='tab:blue', label='other', alpha=.6)

    # Distribution for positive class (should be near 1 ideally)
    sns.histplot(prob_pos, bins=30, kde=True, color='tab:orange', label=cls, alpha=.6)

    plt.title(f'Predicted probability distribution – {cls}')
    plt.xlabel('Predicted probability')
    plt.ylabel('Count')
    plt.legend()
    plt.tight_layout()
    plt.show()

For classes like 45 and 65, the curves are clearly separated, reflecting strong model confidence, while class 73 shows noticeable overlap


# Model Performance Summary

## 1. Overall Picture
- **Class 45 & 65**: Essentially solved.
  - AUC ≈ 1
  - Perfect precision & recall
  - Practically no meaningful errors
- **Class 73**: Bottleneck
  - ROC-AUC: 0.888 (still good but below others)
  - Recall: 69% → 3 out of 10 positives are missed
  - Precision: 73% → 1 in 4 predicted positives is a false alarm
    -
This difficulty is consistent with the 2D PCA plot, where class_45 and class_73 appear closely clustered, and with boxplots, which show overlapping feature distributions. These overlaps likely explain why the model struggles more with class_73.
---

## 2. Performance Metrics

| Class    | Accuracy | Precision | Recall | F1    | ROC-AUC |
|----------|---------|-----------|--------|-------|---------|
| 45       | 0.996   | 0.996     | 0.991  | 0.993 | 0.999   |
| 65       | 1.000   | 1.000     | 1.000  | 1.000 | 1.000   |
| 73       | 0.814   | 0.735     | 0.694  | 0.713 | 0.888   |

---

## 3. Interpretation of One-vs-Rest Confusion Matrices

### 1. Class 45 vs Other
- **True Positives (TP):** 991
- **False Negatives (FN):** 9 → very few 45s missed (≈ 0.9%)
- **False Positives (FP):** 4 → very few non-45s incorrectly predicted as 45
- **True Negatives (TN):** 1996
- **Insight:** Almost perfect classification; recall and precision are extremely high. Model is confident and rarely confused class 45 with others.

### 2. Class 65 vs Other
- **TP:** 2000
- **FN:** 0 → perfect recall
- **FP:** 0 → perfect precision
- **TN:** 2000
- **Insight:** Fully solved. The model separates class 65 from all others flawlessly.

### 3. Class 73 vs Other
- **TP:** 694
- **FN:** 306 → recall ≈ 69% (about 3 out of 10 positives missed)
- **FP:** 251 → precision ≈ 73% (about 1 in 4 predicted positives is a false alarm)
- **TN:** 1749
- **Insight:** Class 73 is the bottleneck. Large number of misclassifications into “other” and some false positives reduce performance. ROC-AUC (≈0.888) is still decent, but the model is less confident here. Most errors arise from uncertain predictions in the 0.4–0.7 probability range.

### Overall Summary
- **Classes 45 & 65:** Essentially solved; almost perfect precision, recall, and AUC.
- **Class 73:** Needs improvement; lower recall and precision due to overlap with other classes. Model struggles with uncertain samples, which is the main source of errors.

---

## 4. Precision-Recall

| Class | Average Precision (AP) | Notes |
|-------|-----------------------|-------|
| 45    | ≈ 0.997               | Almost flawless |
| 65    | 1.000                 | Perfect |
| 73    | ≈ 0.79                | ~10 points below ROC-AUC, typical for imbalanced positives |

---

## 5. Probability Calibration (Histograms)

- **Class 45 & 65**:
  - Peaks well separated (≈ 0 vs. ≈ 1)
  - Model confidence is high and correct
- **Class 73**:
  - Heavy overlap around 0.4–0.7
  - Many samples in the "uncertain" zone → most errors originate here

### Trees

In [ ]:

# Split dataset

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.tree import plot_tree, export_text
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

constant_features = {'is_dance_hit','echo_constant'}
data_set = df.copy()
y = data_set['target_class']
X = data_set.drop(columns=["target_class"]+ list(constant_features))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

#### Decision Tree

In [ ]:
# Basic Decision Tree (no tuning)
dt = DecisionTreeClassifier(random_state=42)  # default hyperparameters
dt.fit(X_train, y_train)

# Evaluate
train_acc = dt.score(X_train, y_train)
test_acc = dt.score(X_test, y_test)

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)

+ The train accuracy is 100%, which means the Decision Tree perfectly memorized the training data.
+ The test accuracy is 75.2%, which is noticeably lower than the training accuracy.

This large gap indicates overfitting: the tree is too complex and captures noise in the training set rather than general patterns.

In [ ]:
# Create a new figure for the plot with a specific size (width=20, height=10 inches)
plt.figure(figsize=(20, 10))

# Plot the trained Decision Tree model
plot_tree(
    dt,                         # Trained Decision Tree classifier
    feature_names=X.columns,    # Names of the input features (from the DataFrame)
    class_names=[str(c) for c in y.unique()],  # Class labels converted to strings
    filled=True,                # Fill nodes with colors to indicate class purity
    rounded=True,               # Use rounded corners for tree nodes
    fontsize=10                 # Set font size for readability
)

# Display the plotted Decision Tree
plt.show()

+ The tree is very deep, which explains why the train accuracy is 100% but the test accuracy is lower (74.2%) — the tree has memorized the training data (overfitting).


In [ ]:
# Predict the class labels for the test dataset using the trained Decision Tree
y_pred = dt.predict(X_test)

# Compute the confusion matrix using true and predicted labels
cm = confusion_matrix(
    y_test,            # True labels
    y_pred,            # Predicted labels
    labels=dt.classes_ # Ensure consistent class ordering
)

# Create a ConfusionMatrixDisplay object for visualization
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=dt.classes_
)

# Plot the confusion matrix using a blue color map
disp.plot(cmap=plt.cm.Blues)

# Add a title to the confusion matrix plot
plt.title("Confusion Matrix - Decision Tree")

# Display the confusion matrix plot
plt.show()

# Generate a detailed classification report (precision, recall, F1-score, support)
report = classification_report(
    y_test,  # True labels
    y_pred   # Predicted labels
)

# Print the classification report to the console
print("Classification Report:\n", report)

+ The tree is overfitting, capturing training data perfectly but generalizing less well to unseen data.

+ class_65 is predicted very well, while class_45 and class_73 are harder to classify due to overlapping feature patterns.

To improve, hyperparameter tuning (max depth, min samples per leaf, etc.) is needed to reduce overfitting.

##### Decision Tree Classifier Hyperparameter tuning


In [ ]:
# Parameters selected for tuning
params = {
    'max_depth': [2, 3, 5, 10, 20],            # limits tree depth; small values reduce overfitting, large values allow complex splits
    'min_samples_leaf': [5, 10, 20, 50, 100],  # minimum samples per leaf; larger values simplify tree and improve generalization
    'criterion': ["gini", "entropy"],          # measures impurity; testing both helps choose best for our dataset
    'splitter':("best", "random")              # "best" selects optimal split, "random" adds variability to reduce overfitting
}

# Build GridSearchCV
tree_cv = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    params,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
    cv=5  # 5-fold cross-validation ensures robust performance evaluation
)

In [ ]:
#Fit tree
tree_cv.fit(X_train, y_train)
best_params = tree_cv.best_params_
best_score = tree_cv.best_score_
best_dt = tree_cv.best_estimator_

#Print results
print(f"Best paramters: {best_params})")
print(f"Best CV score : {best_score})")
print(f"Best Estimator : {best_dt})")

Limiting depth and leaf size reduced overfitting compared to the basic tree.
The model now generalizes better, balancing training performance and test accuracy.

The tuned tree is simpler, avoids memorizing noise, and focuses on the most predictive features.

In [ ]:
# Accuracy
train_acc = best_dt.score(X_train, y_train)
test_acc = best_dt.score(X_test, y_test)
print("Train accuracy after tuning:", train_acc)
print("Test accuracy after tuning:", test_acc)

+ The train and test accuracies are very close, indicating the model no longer overfits the training data.

+ Limiting max_depth and min_samples_leaf helped the tree generalize better.

Overall, the tuned Decision Tree achieves a balanced performance, capturing the main patterns in the data without memorizing noise.

In [ ]:
# Predict class labels on the test set using the tuned Decision Tree
y_pred = best_dt.predict(X_test)

# Compute and display the confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=best_dt.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_dt.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Tuned Decision Tree")
plt.show()

# Print the classification report (precision, recall, F1-score)
print("Classification Report:\n", classification_report(y_test, y_pred))

+ class_45: The model predicts this class fairly precisely, but misses many true class_45 instances.

+ class_65: Excellent performance; the model rarely misclassifies this class.

+ class_73: Decent performance, but some false positives lower precision.

In [ ]:
#Draw Tree
plt.figure(figsize=(20,10))
plot_tree(
    best_dt,
    feature_names=X.columns,
    class_names=[str(c) for c in y.unique()],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.show()

The tuned decision tree has fewer leaf nodes compared to the initial model, indicating a simpler structure. By reducing the number of leaves, the model avoids excessive splitting of the data, which helps prevent overfitting.

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train a baseline Random Forest model with default parameters
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Evaluate training and testing accuracy
train_acc_rf = rf.score(X_train, y_train)
test_acc_rf = rf.score(X_test, y_test)
print("RF train acc:", train_acc_rf)
print("RF test acc:", test_acc_rf)

# Generate predictions and print classification report
y_pred_rf = rf.predict(X_test)
print("Classification Report (RF baseline):\n", classification_report(y_test, y_pred_rf))

The model performs well overall (77–78% accuracy) but:
 + Class 65 dominates performance (very high precision/recall).
 + Class 45 and 73 are harder to classify, especially class 73 (lower recall).

Overfitting is clearly present because:
+ Train accuracy = 100%
+ Test accuracy = 77.5%

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf, labels=rf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=rf.classes_)
fig, ax = plt.subplots(figsize=(8,6))
disp.plot(ax=ax, cmap=plt.cm.Blues)
plt.title("Random Forest - Confusion Matrix (baseline)")
plt.show()

In [ ]:
# Compute and sort feature importances from the Random Forest model
importances = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

# Print feature importance scores
print("RF feature importances (baseline):\n")
print(importances)

# Plot all feature importances
plt.figure(figsize=(10, 6))
importances.plot(kind='barh')
plt.gca().invert_yaxis()   # Most important features at the top
plt.title("Random Forest – Feature Importances (All Features)")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

+ purity_score is again the strongest predictor → matches  DT results.
+ energy/loudness/signal features are now much more relevant than in the Decision Tree.
+ Some movement/timbre/musicological features also matter.

Many features have very low or zero importance (expected for RF with many correlated variables).
This shows RF captures nonlinear interactions much better than the Decision Tree.

##### Hyperparameter tuning for Random Forest

In [ ]:
# Hyperparameter grid for tuning the Random Forest
params_rf = {
    'n_estimators': [50, 75, 100],      # Number of trees
    'max_depth': [8, 10, 12],           # Limit tree depth to reduce overfitting
    'min_samples_leaf': [30, 40, 50],   # Enforce larger leaf sizes
    'max_features': ['sqrt', 0.15],     # Control feature randomness
    'class_weight': ['balanced'],       # Handle class imbalance
}

# Grid search with cross-validation
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=params_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Fit grid search on training data
rf_grid.fit(X_train, y_train)

# Print best hyperparameters and cross-validation score
print("Best RF params:", rf_grid.best_params_)
print("Best CV score:", rf_grid.best_score_)

# Retrieve the best Random Forest model
best_rf = rf_grid.best_estimator_

# Predict on test data using the tuned Random Forest
y_pred_best_rf = best_rf.predict(X_test)

# Evaluate tuned Random Forest performance
print("Train acc (best RF):", best_rf.score(X_train, y_train))
print("Test acc (best RF):", best_rf.score(X_test, y_test))
print("Classification Report (best RF):\n", classification_report(y_test, y_pred_best_rf))

+ Train Accuracy: 0.820
+ Test Accuracy: 0.776
+ The tuned Random Forest achieved a training accuracy of 82.0% and a test accuracy of 77.7%, showing good generalization with minimal overfitting. The model performs particularly well on class_65, achieving high precision and recall, while class_45 and class_73 show slightly lower but balanced performance. Overall, the macro F1-score of 0.77 indicates that the model reliably distinguishes between the classes without favoring any particular one.

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best_rf, labels=best_rf.classes_)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=best_rf.classes_)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap=plt.cm.Blues, colorbar=True)
plt.title("Random Forest (Tuned) - Confusion Matrix")
plt.show()

In [ ]:
# Compute feature importances from the tuned Random Forest
rf_importances = pd.Series(best_rf.feature_importances_, index=X.columns)
rf_importances_sorted = rf_importances.sort_values(ascending=False)

# Print all feature importances
print("Random Forest - ALL Feature Importances:")
print(rf_importances_sorted)

# Plot all feature importances
plt.figure(figsize=(10, 6))
rf_importances_sorted.plot(kind='barh')
plt.gca().invert_yaxis()  # Most important features at the top
plt.title("Random Forest - Feature Importances (ALL Features)")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

Feature importances became more concentrated

+ purity_score increased from 0.116 → 0.1936

+ target_regression increased from 0.067 → 0.0927

This means RF tuned found a more stable splitting pattern.

Some features gained importance

Examples:

+ loud_energy_ratio (from 0.067 → 0.082)

+ energy_rank_pct (from 0.037 → 0.074)

+ signal_power / signal_strength increased

These reflect deeper interactions captured by the tuned RF.

Many previously useless features still remain at zero

(e.g., time_signature, dance_hit, echo_constant)

This is expected: RF filters out noisy predictors.


# Random Forest Before vs After Tuning

The baseline Random Forest achieved **high training accuracy (1.00)** but a significantly lower **test accuracy (0.775)**, indicating clear **overfitting**. After hyperparameter tuning, the overfitting was substantially reduced:

* **Train accuracy dropped** to **0.820**,
* **Test accuracy increased** to **0.776**,
* Cross-validation score also improved (**0.7841**),

showing that the tuned model generalizes better and is more stable across folds.

### **Class-level performance**

Across both models, class **65** consistently shows the highest precision/recall, while **class 45 and 73** remain harder to separate.
After tuning, class 73 improves slightly (f1 from ~0.68 → ~0.72), suggesting that the tuned tree structure allows the model to capture more subtle patterns for this class.

---

### **Random Forest – Feature Importance Analysis**

**Before Tuning:**

* Feature importance was fairly spread out among many predictors.
* Top features included `purity_score` (0.117), `signal_strength` (0.081), `loud_energy_ratio` (0.068), and `target_regression` (0.067).
* Many features had very low or zero importance, indicating the baseline model relied heavily on a subset of predictors but still considered some weaker features.

**After Tuning:**

* The improved RF shifted more importance to the strongest features: `purity_score` (0.194), `target_regression` (0.093), `loud_energy_ratio` (0.082), and `energy_rank_pct` (0.074).
* The distribution is now more focused on the most predictive variables, while weaker features contribute less.

**Interpretation:**

* Hyperparameter tuning allowed the RF to **prioritize the most informative features**, improving generalization.
* This reduced overfitting compared to the baseline, as the model now relies less on weaker predictors and focuses on variables with the strongest signal.


In [ ]:
importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top_features = importances.head(10)  # top 10 for visualization
print(top_features)

##### Interpretation in Context


### **• Univariate and Bivariate Analysis (Task 2)**

Many of the highly ranked RF features (e.g., **purity_score, loudness/energy features, tempo vs genre, artist popularity-related metrics**) were also among the variables that showed strong separation between classes that has top feature importances.
This reinforces the idea that RF is capturing real signal rather than noise.

### **• Ridge / Lasso (Task 5)**

Features that Lasso retained (strong predictors) overlap heavily with the ones RF now considers important:

* purity_score
* target_regression
* artist popularity metrics
* intensity / energy indicators

The feature importance analysis from the tuned Random Forest aligns closely with the results obtained from the Lasso and Ridge regressions. Both Lasso and Ridge identified `album_freq`, `intensity_level`, and certain duration-related features as highly predictive, which corresponds to the RF’s top features such as `purity_score`, `target_regression`, and `loud_energy_ratio`.

In particular, Lasso’s sparsity highlighted a small subset of strong predictors while shrinking or eliminating redundant features, mirroring how the tuned RF shifted importance toward a few dominant variables. Ridge, which penalizes large coefficients without zeroing them out, similarly emphasizes the most relevant predictors while retaining weaker features, reflecting the more distributed feature importances observed in the baseline RF.

Overall, the convergence of results across these methods suggests that both tree-based and linear-penalized models consistently identify the same core set of informative features. This reinforces their significance for predicting the target and confirms that the tuned Random Forest, like Lasso and Ridge, effectively filters out noise and emphasizes the strongest signals.